In [ ]:
"""
Supertrend Scanner for NSE India (Daily Timeframe)
----------------------------------------------------
Scans a list of NSE stocks and prints which ones are showing
a Supertrend BUY signal — matching the Pine Script logic exactly.

Requirements:
    pip install yfinance pandas ta-lib-python
    OR simply: pip install yfinance pandas

Usage:
    python supertrend_scanner.py
"""

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# ─────────────────────────────────────────────
# CONFIG — tweak these to match your Pine inputs
# ─────────────────────────────────────────────
ATR_PERIOD   = 10
MULTIPLIER   = 3.0
LOOKBACK     = 200   # candles to download (more = more accurate ATR warmup)

# ─────────────────────────────────────────────
# NSE STOCK LIST  (add/remove tickers here)
# Suffix ".NS" is required for NSE via yfinance
# ─────────────────────────────────────────────
STOCKS = [
    # Nifty 50 core
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS",
    "HINDUNILVR.NS", "SBIN.NS", "BAJFINANCE.NS", "BHARTIARTL.NS", "KOTAKBANK.NS",
    "LT.NS", "AXISBANK.NS", "ITC.NS", "ASIANPAINT.NS", "MARUTI.NS",
    "SUNPHARMA.NS", "TITAN.NS", "ULTRACEMCO.NS", "WIPRO.NS", "HCLTECH.NS",
    "TECHM.NS", "POWERGRID.NS", "NTPC.NS", "ONGC.NS", "COALINDIA.NS",
    "ADANIENT.NS", "ADANIPORTS.NS", "JSWSTEEL.NS", "TATASTEEL.NS", "TATAMOTORS.NS",
    "M&M.NS", "BAJAJFINSV.NS", "GRASIM.NS", "HINDALCO.NS", "INDUSINDBK.NS",
    "SBILIFE.NS", "HDFCLIFE.NS", "DIVISLAB.NS", "CIPLA.NS", "DRREDDY.NS",
    "BPCL.NS", "HEROMOTOCO.NS", "EICHERMOT.NS", "APOLLOHOSP.NS", "TATACONSUM.NS",
    "BRITANNIA.NS", "NESTLEIND.NS", "UPL.NS", "SHREECEM.NS", "PIDILITIND.NS",
    # Midcap extras (optional — add more as you like)
    "MUTHOOTFIN.NS", "CHOLAFIN.NS", "PERSISTENT.NS", "COFORGE.NS", "LTIM.NS",
    "IRCTC.NS", "HAL.NS", "BEL.NS", "PAGEIND.NS", "VOLTAS.NS",
]

# ─────────────────────────────────────────────
# SUPERTREND CALCULATION
# Mirrors the Pine Script logic exactly:
#   up  = src - (multiplier * atr)   then ratcheted up
#   dn  = src + (multiplier * atr)   then ratcheted down
#   trend flips when close crosses the band
# ─────────────────────────────────────────────
def compute_supertrend(df: pd.DataFrame, period: int = 10, multiplier: float = 3.0):
    hl2  = (df["High"] + df["Low"]) / 2
    close = df["Close"]

    # True Range
    prev_close = close.shift(1)
    tr = pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"]  - prev_close).abs(),
    ], axis=1).max(axis=1)

    # ATR via Wilder's RMA (same as Pine's atr())
    atr = tr.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()

    up_raw = hl2 - (multiplier * atr)
    dn_raw = hl2 + (multiplier * atr)

    up    = up_raw.copy()
    dn    = dn_raw.copy()
    trend = pd.Series(np.ones(len(df), dtype=int), index=df.index)

    for i in range(1, len(df)):
        # Ratchet up band
        up.iloc[i]  = max(up_raw.iloc[i], up.iloc[i-1])  if close.iloc[i-1] > up.iloc[i-1]  else up_raw.iloc[i]
        # Ratchet down band
        dn.iloc[i]  = min(dn_raw.iloc[i], dn.iloc[i-1])  if close.iloc[i-1] < dn.iloc[i-1]  else dn_raw.iloc[i]

        # Trend flip logic
        if trend.iloc[i-1] == -1 and close.iloc[i] > dn.iloc[i-1]:
            trend.iloc[i] = 1
        elif trend.iloc[i-1] == 1 and close.iloc[i] < up.iloc[i-1]:
            trend.iloc[i] = -1
        else:
            trend.iloc[i] = trend.iloc[i-1]

    buy_signal  = (trend == 1) & (trend.shift(1) == -1)   # just crossed up
    sell_signal = (trend == -1) & (trend.shift(1) == 1)   # just crossed down

    return trend, up, dn, buy_signal, sell_signal


# ─────────────────────────────────────────────
# SCANNER
# ─────────────────────────────────────────────
def scan():
    print(f"\n{'='*60}")
    print(f"  SUPERTREND NSE SCANNER  |  Daily  |  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f"  ATR Period: {ATR_PERIOD}  |  Multiplier: {MULTIPLIER}")
    print(f"{'='*60}\n")

    buy_stocks   = []
    sell_stocks  = []
    uptrend_stocks = []
    errors       = []

    for ticker in STOCKS:
        try:
            df = yf.download(
                ticker,
                period=f"{LOOKBACK}d",
                interval="1d",
                progress=False,
                auto_adjust=True,
            )

            if df is None or len(df) < ATR_PERIOD + 5:
                errors.append((ticker, "Not enough data"))
                continue

            df.dropna(inplace=True)

            # Flatten MultiIndex columns if present (yfinance quirk)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            trend, up, dn, buy_sig, sell_sig = compute_supertrend(df, ATR_PERIOD, MULTIPLIER)

            last_trend    = trend.iloc[-1]
            last_buy      = buy_sig.iloc[-1]
            last_sell     = sell_sig.iloc[-1]
            last_close    = df["Close"].iloc[-1]

            # Freshly flipped to BUY today
            if last_buy:
                buy_stocks.append({
                    "Ticker": ticker,
                    "Close": round(last_close, 2),
                    "Signal": "🟢 NEW BUY",
                })

            # Freshly flipped to SELL today
            elif last_sell:
                sell_stocks.append({
                    "Ticker": ticker,
                    "Close": round(last_close, 2),
                    "Signal": "🔴 NEW SELL",
                })

            # Already in uptrend (not a new signal, but riding the trend)
            elif last_trend == 1:
                uptrend_stocks.append({
                    "Ticker": ticker,
                    "Close": round(last_close, 2),
                    "Signal": "🟡 UPTREND",
                })

        except Exception as e:
            errors.append((ticker, str(e)))

    # ── Print Results ──────────────────────────────────────────
    section("🟢 NEW BUY SIGNALS (trend flipped today)", buy_stocks)
    section("🔴 NEW SELL SIGNALS (trend flipped today)", sell_stocks)
    section("🟡 ALREADY IN UPTREND (riding buy)", uptrend_stocks)

    if errors:
        print(f"\n⚠️  Errors / Skipped ({len(errors)}):")
        for t, msg in errors:
            print(f"   {t:25s}  {msg}")

    print(f"\n{'='*60}\n")


def section(title: str, rows: list):
    print(f"\n{title}  ({len(rows)} stocks)")
    print("-" * 45)
    if not rows:
        print("   None")
    else:
        print(f"  {'Ticker':<20} {'Close':>10}   Signal")
        print(f"  {'-'*18} {'-'*10}   {'-'*12}")
        for r in rows:
            print(f"  {r['Ticker']:<20} {r['Close']:>10.2f}   {r['Signal']}")


# ─────────────────────────────────────────────
if __name__ == "__main__":
    scan()

In [2]:
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import json
import os
import re
from datetime import datetime

print("="*70)
print("     SWING TRADING AUTO SCREENER - FULLY DYNAMIC")
print("="*70)

# Stock ID cache file
ID_CACHE_FILE = "stock_ids.json"
STOCKEDGE_ID_CACHE_FILE = "stockedge_ids.json"

# Load existing IDs from cache
if os.path.exists(ID_CACHE_FILE):
    with open(ID_CACHE_FILE, 'r') as f:
        STOCK_ID_MAP = json.load(f)
    print(f"📂 Loaded {len(STOCK_ID_MAP)} Trendlyne IDs from cache")
else:
    # Initial paths from your notebook (ID/SYMBOL/company-name)
    STOCK_ID_MAP = {
        'HEROMOTOCO': {'id': '540', 'path': '540/HEROMOTOCO/hero-motocorp-ltd'},
        # ... (keep existing initial map if needed, or just start empty/from cache)
    }
    # For brevity in this edit, I'll assume the file usually exists or the user is fine with the existing hardcoded list being the start.
    # The original code had a big list here. I will leave it as is in the file, just updating the cache loading part.

if os.path.exists(STOCKEDGE_ID_CACHE_FILE):
    with open(STOCKEDGE_ID_CACHE_FILE, 'r') as f:
        STOCKEDGE_ID_MAP = json.load(f)
    print(f"📂 Loaded {len(STOCKEDGE_ID_MAP)} StockEdge IDs from cache")
else:
    STOCKEDGE_ID_MAP = {}

# Load existing IDs from cache
if os.path.exists(ID_CACHE_FILE):
    with open(ID_CACHE_FILE, 'r') as f:
        STOCK_ID_MAP = json.load(f)
    print(f"📂 Loaded {len(STOCK_ID_MAP)} stock IDs from cache")
else:
    # Initial paths from your notebook (ID/SYMBOL/company-name)
    STOCK_ID_MAP = {
        'HEROMOTOCO': {'id': '540', 'path': '540/HEROMOTOCO/hero-motocorp-ltd'},
        'BSE': {'id': '52884', 'path': '52884/BSE/bse-ltd'},
        'MUTHOOTFIN': {'id': '897', 'path': '897/MUTHOOTFIN/muthoot-finance-ltd'},
        'GMRAIRPORT': {'id': '472', 'path': '472/GMRAIRPORT/gmr-airports-ltd'},
        'ASIANPAINT': {'id': '117', 'path': '117/ASIANPAINT/asian-paints-ltd'},
        '360ONE': {'id': '166089', 'path': '166089/360ONE/360-one-wam-ltd'},
        'ANGELONE': {'id': '300864', 'path': '300864/ANGELONE/angel-one-ltd'},
        'MOTHERSON': {'id': '878', 'path': '878/MOTHERSON/samvardhana-motherson-international-ltd'},
        'BHARTIARTL': {'id': '187', 'path': '187/BHARTIARTL/bharti-airtel-ltd'},
        'LTIM': {'id': '4596', 'path': '4596/LTIM/ltimindtree-ltd'},
        'BAJAJFINSV': {'id': '147', 'path': '147/BAJAJFINSV/bajaj-finserv-ltd'},
        'PERSISTENT': {'id': '1023', 'path': '1023/PERSISTENT/persistent-systems-ltd'},
        'AXISBANK': {'id': '140', 'path': '140/AXISBANK/axis-bank-ltd'},
        'CANBK': {'id': '232', 'path': '232/CANBK/canara-bank'},
        'HCLTECH': {'id': '531', 'path': '531/HCLTECH/hcl-technologies-ltd'},
        'FEDERALBNK': {'id': '412', 'path': '412/FEDERALBNK/federal-bank-ltd'},
        'MCX': {'id': '853', 'path': '853/MCX/multi-commodity-exchange-of-india-ltd'},
        'RELIANCE': {'id': '1127', 'path': '1127/RELIANCE/reliance-industries-ltd'},
        'EICHERMOT': {'id': '360', 'path': '360/EICHERMOT/eicher-motors-ltd'},
        'SONACOMS': {'id': '547977', 'path': '547977/SONACOMS/sona-blw-precision-forgings-ltd'},
        'SUNPHARMA': {'id': '1316', 'path': '1316/SUNPHARMA/sun-pharmaceutical-industries-ltd'},
        'AUROPHARMA': {'id': '132', 'path': '132/AUROPHARMA/aurobindo-pharma-ltd'},
        'AUBANK': {'id': '54902', 'path': '54902/AUBANK/au-small-finance-bank-ltd'},
        'SBIN': {'id': '1193', 'path': '1193/SBIN/state-bank-of-india'},
        'HUDCO': {'id': '54143', 'path': '54143/HUDCO/housing-and-urban-development-corporation-ltd'},
        'COFORGE': {'id': '942', 'path': '942/COFORGE/coforge-ltd'},
        'MANAPPURAM': {'id': '827', 'path': '827/MANAPPURAM/manappuram-finance-ltd'},
        'BIOCON': {'id': '197', 'path': '197/BIOCON/biocon-ltd'},
        'KEI': {'id': '729', 'path': '729/KEI/kei-industries-ltd'},
        'NYKAA': {'id': '705188', 'path': '705188/NYKAA/fsn-e-commerce-ventures-ltd'},
        'POWERINDIA': {'id': '202203', 'path': '202203/POWERINDIA/hitachi-energy-india-ltd'},
        'BHARATFORG': {'id': '184', 'path': '184/BHARATFORG/bharat-forge-ltd'},
        'TITAN': {'id': '1403', 'path': '1403/TITAN/titan-company-ltd'},
        'BAJFINANCE': {'id': '150', 'path': '150/BAJFINANCE/bajaj-finance-ltd'},
        'NBCC': {'id': '916', 'path': '916/NBCC/nbcc-india-ltd'},
        'LT': {'id': '800', 'path': '800/LT/larsen-toubro-ltd'},
        'ICICIGI': {'id': '61147', 'path': '61147/ICICIGI/icici-lombard-general-insurance-company-ltd'}
    }
    print(f"📂 Initialized with {len(STOCK_ID_MAP)} stock IDs")

def save_stock_ids():
    """Save stock IDs to cache file"""
    with open(ID_CACHE_FILE, 'w') as f:
        json.dump(STOCK_ID_MAP, f, indent=4)
    print(f"💾 Saved {len(STOCK_ID_MAP)} Trendlyne IDs to cache")

def save_stockedge_ids():
    """Save StockEdge IDs to cache file"""
    with open(STOCKEDGE_ID_CACHE_FILE, 'w') as f:
        json.dump(STOCKEDGE_ID_MAP, f, indent=4)
    print(f"💾 Saved {len(STOCKEDGE_ID_MAP)} StockEdge IDs to cache")

def find_stock_id_google_fallback(symbol):
    """Fallback: Search Google for Trendlyne URL"""
    try:
        print(f"   🔄 Trying Google search fallback...")
        search_query = f"site:trendlyne.com/equity {symbol}"
        search_url = f"https://www.google.com/search?q={requests.utils.quote(search_query)}"
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(search_url, headers=headers, timeout=10)
        
        # Simple regex to find Trendlyne equity URLs
        pattern = r'https://trendlyne\.com/equity/(\d+)/([A-Z0-9]+)/([a-z0-9-]+)'
        matches = re.findall(pattern, response.text)
        
        if matches:
            stock_id, symbol_part, company_name = matches[0]
            full_path = f"{stock_id}/{symbol_part}/{company_name}"
            print(f"   ✅ Found via Google: {full_path}")
            return {"id": stock_id, "path": full_path}
        else:
            print(f"   ❌ No results in Google search")
            return None
    except Exception as e:
        print(f"   ❌ Google fallback failed: {str(e)[:60]}")
        return None

def find_stock_id(driver, wait, symbol, max_retries=3):
    """Search Trendlyne and extract stock ID and company path with improved robustness"""
    
    for attempt in range(max_retries):
        try:
            if attempt > 0:
                wait_time = 2 ** attempt  # Exponential backoff: 2, 4, 8 seconds
                print(f"   🔄 Retry {attempt}/{max_retries-1} (waiting {wait_time}s)...")
                time.sleep(wait_time)
            else:
                print(f"   🔍 Searching for {symbol}...")
            
            # Navigate to Trendlyne
            driver.get("https://trendlyne.com/")
            time.sleep(3)  # Increased initial wait
            
            # Try multiple strategies to find the search bar
            search_bar = None
            selectors = [
                (By.XPATH, '(//input[@placeholder="Search Stock, index, MF"])[1]'),
                (By.CSS_SELECTOR, 'input[placeholder*="Search Stock"]'),
                (By.CSS_SELECTOR, 'input.search-input'),
                (By.XPATH, '//input[contains(@placeholder, "Search")]')
            ]
            
            for by, selector in selectors:
                try:
                    search_bar = WebDriverWait(driver, 10).until(
                        EC.visibility_of_element_located((by, selector))
                    )
                    if search_bar:
                        break
                except:
                    continue
            
            if not search_bar:
                raise Exception("Could not locate search bar with any selector")
            
            # Clear and enter search term
            search_bar.clear()
            time.sleep(0.5)
            search_bar.send_keys(symbol)
            time.sleep(4)  # Increased wait for autocomplete
            
            # Wait for autocomplete suggestions to appear
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "ul.ui-menu"))
                )
            except:
                pass  # Continue even if this specific selector doesn't work
            
            # Find ALL links in the autocomplete suggestions
            suggestion_links = []
            suggestion_selectors = [
                (By.CSS_SELECTOR, "ul.ui-menu a[href*='/equity/']"),
                (By.CSS_SELECTOR, "div.autocomplete-suggestion a[href*='/equity/']"),
                (By.CSS_SELECTOR, ".autocomplete-suggestions a[href*='/equity/']"),
                (By.XPATH, '//ul[contains(@class, "ui-menu")]//a[contains(@href, "/equity/")]'),
            ]
            
            for by, selector in suggestion_selectors:
                try:
                    links = driver.find_elements(by, selector)
                    if links:
                        suggestion_links.extend(links)
                        break
                except:
                    continue
            
            if not suggestion_links:
                raise Exception("No autocomplete suggestions found")
            
            # Filter for links that contain the stock symbol in the URL
            stock_url = None
            for link in suggestion_links:
                href = link.get_attribute("href")
                if href and "/equity/" in href and f"/{symbol}/" in href.upper():
                    stock_url = href
                    break
            
            # If no exact match, try the first equity link
            if not stock_url and suggestion_links:
                stock_url = suggestion_links[0].get_attribute("href")
            
            if not stock_url:
                raise Exception("Could not extract stock URL from suggestions")
            
            # Validate the URL format
            if "group-insider-trading" in stock_url or "/equity/" not in stock_url:
                raise Exception(f"Invalid stock URL captured: {stock_url}")
            
            # Extract ID and company path from URL
            # URL format: https://trendlyne.com/equity/12345/SYMBOL/company-name/
            parts = stock_url.split("/")
            if "equity" in parts:
                idx = parts.index("equity")
                stock_id = parts[idx + 1]
                symbol_part = parts[idx + 2]
                company_name = parts[idx + 3] if len(parts) > idx + 3 else ""
                
                # Store full path: ID/SYMBOL/company-name
                full_path = f"{stock_id}/{symbol_part}/{company_name}" if company_name else f"{stock_id}/{symbol_part}"
                
                print(f"   ✅ Found: {full_path}")
                return {"id": stock_id, "path": full_path}
            else:
                print(f"   ❌ Could not parse URL: {stock_url}")
                return None
                
        except Exception as e:
            error_msg = str(e)[:100]
            if attempt < max_retries - 1:
                print(f"   ⚠️  Attempt {attempt + 1} failed: {error_msg}")
            else:
                print(f"   ❌ All Trendlyne attempts failed: {error_msg}")
    
    # If all retries failed, try Google fallback
    return find_stock_id_google_fallback(symbol)

def search_stockedge_for_id(driver, symbol, company_name=None):
    """
    Search StockEdge for a company and extract its correct API ID.
    Uses StockEdge's internal APIs for speed and reliability.
    Returns the DefaultListingID needed for technical indicators.
    """
    cache = load_cache() if 'load_cache' in globals() else {}
    cache_key = f"{symbol}_{company_name}" if company_name else symbol
    
    if cache_key in STOCKEDGE_ID_MAP:
        return STOCKEDGE_ID_MAP[cache_key]

    print(f"   🔍 Searching StockEdge API for {symbol}...")
    
    try:
        # Step 1: Universal Search to get the DocId (the ID used in URLs)
        search_term = company_name.replace('-', ' ') if company_name else symbol
        search_url = f"https://api.stockedge.com/Api/UniversalSearchApi/GetQuickSearchResult?searchTerm={requests.utils.quote(search_term)}&lang=en"
        r = requests.get(search_url, timeout=10)
        data = r.json()
        
        search_results = data.get('Data', [])
        doc_id = None
        for item in search_results:
            # We want the 'se_security' entry (which represents a stock)
            if item.get('EntityCode') == 'se_security':
                doc_id = item.get('DocId')
                break
        
        if doc_id:
            # Step 2: Get Security Info to find the DefaultListingID (the ID used for technical data)
            info_url = f"https://api.stockedge.com/Api/SecurityDashboardApi/GetLatestSecurityInfo/{doc_id}?lang=en"
            r_info = requests.get(info_url, timeout=10)
            info_data = r_info.json()
            
            listing_id = info_data.get('DefaultListingID')
            if listing_id:
                found_id = str(listing_id)
                STOCKEDGE_ID_MAP[cache_key] = found_id
                save_stockedge_ids()
                print(f"   ✅ Found StockEdge ID: {found_id}")
                return found_id

    except Exception as e:
        print(f"   ⚠️ API search failed for '{symbol}': {e}")

    return None

def fetch_stockedge_indicators(stock_id):
    """Fetch technical indicators from StockEdge API using the DefaultListingID."""
    if not stock_id:
        return None
        
    url = f"https://api.stockedge.com/Api/SecurityDashboardApi/GetTechnicalIndicators/{stock_id}?lang=en"
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Referer": "https://web.stockedge.com/"
        }
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.RequestException as e:
        print(f"   ❌ API error for ID {stock_id}: {e}")
        return None

def parse_stockedge_data(json_data):
    """Parse the StockEdge API JSON response to extract RSI and Relative Strength indicators."""
    if not json_data:
        return {}, "Not Available"
    
    indicators_needed = {
        "Relative Strength Index (Daily)": "Relative Strength Index (Daily)",
        "Relative Strength Index (Weekly)": "Relative Strength Index (Weekly)",
        "Relative Strength Benchmark Index (21 Days)": "Relative Strength Benchmark Index (21 Days)",
        "Relative Strength Benchmark Index (55 Days)": "Relative Strength Benchmark Index (55 Days)",
        "Relative Strength Benchmark Index (21 Weeks)": "Relative Strength Benchmark Index (21 Weeks)",
        "Static Relative Strength": "Static Relative Strength",
        "Adaptive Relative Strength": "Adaptive Relative Strength",
        "Relative Strength Sector Index (55 Days)": "Relative Strength Sector Index (55 Days)"
    }
    
    se_data = {}
    rs_positive_count = 0
    
    if isinstance(json_data, list):
        for item in json_data:
            name = item.get("Name")
            if name in indicators_needed:
                val = item.get("Value", "N/A")
                desc = item.get("Desc") or ""
                se_data[name] = {"value": val, "status": desc}
                
                # Count positive indicators for RS keys only
                if name in [
                    "Relative Strength Benchmark Index (21 Days)",
                    "Relative Strength Benchmark Index (55 Days)",
                    "Relative Strength Benchmark Index (21 Weeks)",
                    "Static Relative Strength",
                    "Adaptive Relative Strength",
                    "Relative Strength Sector Index (55 Days)"
                ]:
                    if "Positive" in desc:
                        rs_positive_count += 1
    
    return se_data, rs_positive_count

# --------------------------
# 1️⃣ Fetch stocks from API
# --------------------------
url = "https://intradayscreener.com/api/indicatorscans/reloutperf/OUTPERFORM_SHORT_MEDIUM?filter=fno"

try:
    print("\n📡 Fetching momentum stocks from API...")
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    if isinstance(data, list) and all(isinstance(item, dict) for item in data):
        df = pd.DataFrame(data)
    else:
        print("\nData is not in expected list-of-dicts format.")
        exit()

except requests.exceptions.RequestException as e:
    print(f"❌ Error fetching data: {e}")
    exit()

data = df[df['scanDescription'] == 'Outperforming benchmark short and medium term']
stock = data['symbol'].tolist()

# Filter out index symbols
skip_stocks = ['NIFTY_MID_SELECT', 'NIFTY_NEXT_50', 'NIFTY_FIN_SERVICE', 'NIFTY BANK']
stock = [s for s in stock if s not in skip_stocks]

print(f"✅ Found {len(stock)} momentum stocks from API")

# Check which stocks need IDs
stocks_with_ids = [s for s in stock if s in STOCK_ID_MAP]
stocks_without_ids = [s for s in stock if s not in STOCK_ID_MAP]

print(f"📋 Have IDs for: {len(stocks_with_ids)} stocks")
if stocks_without_ids:
    print(f"🔍 Need to find IDs for: {len(stocks_without_ids)} stocks")

# --------------------------
# 2️⃣ Setup Selenium
# --------------------------
print("\n🌐 Setting up Chrome browser...")
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--log-level=3")  # Suppress console messages
chrome_options.add_experimental_option('excludeSwitches', ['enable-logging'])  # Suppress DevTools messages
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
wait = WebDriverWait(driver, 15)

# --------------------------
# 2.5️⃣ Find missing IDs
# --------------------------
if stocks_without_ids:
    print(f"\n{'='*70}")
    print(f"🔍 Finding IDs for {len(stocks_without_ids)} new stocks...")
    print(f"{'='*70}")
    
    for i, symbol in enumerate(stocks_without_ids):
        print(f"\n[{i+1}/{len(stocks_without_ids)}] {symbol}")
        stock_id = find_stock_id(driver, wait, symbol)
        if stock_id:
            STOCK_ID_MAP[symbol] = stock_id
            stocks_with_ids.append(symbol)
    
    # Save updated IDs
    save_stock_ids()
    print(f"\n✅ Now have IDs for {len(stocks_with_ids)} stocks")

financial_results = []
delivery_results = []
skipped_bad_status = []
failed_stocks = []

total_stocks = len(stocks_with_ids)
start_time = time.time()

# --------------------------
# 3️⃣ Process each stock
# --------------------------
print(f"\n{'='*70}")
print(f"🚀 Starting to process {total_stocks} stocks...")
print(f"{'='*70}")

try:
    for i, symbol in enumerate(stocks_with_ids):
        processed = i + 1
        remaining = total_stocks - processed
        
        # Calculate ETA
        if processed > 1:
            elapsed = time.time() - start_time
            avg_time = elapsed / (processed - 1)
            est_remaining = avg_time * remaining
            eta_min = int(est_remaining // 60)
            eta_sec = int(est_remaining % 60)
            eta_str = f"~{eta_min}m {eta_sec}s"
        else:
            eta_str = "calculating..."
        
        print(f"\n{'─'*70}")
        print(f"🔎 [{processed}/{total_stocks}] {symbol}")
        print(f"   ⏱️  Remaining: {remaining} stocks | ETA: {eta_str}")
        
        try:
            # Build URLs directly using the full path
            stock_data = STOCK_ID_MAP[symbol]
            stock_path = stock_data['path'] if isinstance(stock_data, dict) else stock_data
            
            fin_url = f"https://trendlyne.com/fundamentals/financials/{stock_path}/"
            del_url = f"https://trendlyne.com/equity/delivery-analysis/{stock_path}/"
            
            # Scrape Financials
            print(f"   📊 Checking revenue...")
            driver.get(fin_url)
            time.sleep(2)
            
            table = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table")))
            headers = [th.text.strip() for th in table.find_elements(By.TAG_NAME, "th")]
            rows_data = []
            for row in table.find_elements(By.TAG_NAME, "tr")[1:]:
                cells = [td.text.strip() for td in row.find_elements(By.TAG_NAME, "td")]
                if cells:
                    rows_data.append(cells)
            
            df_fin = pd.DataFrame(rows_data, columns=headers)
            
            # Check Revenue
            total_rev_row = df_fin[df_fin['Indicator'] == 'Total Rev.']
            if not total_rev_row.empty:
                quarter_cols = [col for col in total_rev_row.columns if "'" in col]
                rev_values = total_rev_row[quarter_cols].iloc[0]
                rev_values = rev_values.replace('', '0').replace(',', '', regex=True).astype(float)
                
                latest_q = rev_values.index[0]
                latest_v = rev_values.values[0]
                prev_q = rev_values.index[1]
                prev_v = rev_values.values[1]
                
                status = "Good ↑" if latest_v > prev_v else "Bad ↓"
                change_pct = ((latest_v - prev_v) / prev_v * 100) if prev_v != 0 else 0
                
                if status == "Good ↑":
                    print(f"   ✅ GOOD: Revenue {change_pct:+.1f}% ({prev_v:.0f} → {latest_v:.0f})")
                    
                    financial_results.append({
                        "Stock": symbol,
                        "Latest Quarter": latest_q,
                        "Latest Revenue": latest_v,
                        "Previous Quarter": prev_q,
                        "Previous Revenue": prev_v,
                        "Change": latest_v - prev_v,
                        "Change %": change_pct,
                        "Status": status
                    })
                    
                    # Scrape Delivery
                    print(f"   📦 Getting delivery data...")
                    try:
                        driver.get(del_url)
                        time.sleep(3)  # Increased wait time
                        
                        # Wait for table with longer timeout
                        wait_delivery = WebDriverWait(driver, 20)
                        table = wait_delivery.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table")))
                        
                        headers = [th.text.strip() for th in table.find_elements(By.TAG_NAME, "th")]
                        rows_data = []
                        for row in table.find_elements(By.TAG_NAME, "tr")[1:]:
                            cells = [td.text.strip() for td in row.find_elements(By.TAG_NAME, "td")]
                            if cells:
                                rows_data.append(cells)
                        
                        if rows_data:
                            df_del = pd.DataFrame(rows_data, columns=headers)
                            df_del.insert(0, "Stock", symbol)
                            delivery_results.append(df_del)
                            print(f"   ✅ Delivery data collected")
                        else:
                            print(f"   ⚠️  No delivery rows found")
                    except Exception as e:
                        print(f"   ⚠️  Delivery data unavailable (continuing...)")
                    
                    # --------------------------
                    # Scrape StockEdge (Using new API-based approach)
                    # --------------------------
                    # Extract company name from Trendlyne path for better search
                    # Path format: id/SYMBOL/company-name-slug
                    company_name_slug = None
                    if stock_path:
                        parts = stock_path.split("/")
                        if len(parts) >= 3:
                            company_name_slug = parts[2]
                    
                    # Get StockEdge ID (DefaultListingID)
                    se_id = search_stockedge_for_id(driver, symbol, company_name=company_name_slug)
                    
                    se_data = {}
                    rsi_status = "Not Available"
                    
                    if se_id:
                        print(f"   📊 Getting StockEdge indicators...")
                        json_data = fetch_stockedge_indicators(se_id)
                        
                        if json_data:
                            se_data, rs_positive_count = parse_stockedge_data(json_data)
                            rsi_status = rs_positive_count
                            
                            if len(se_data) > 0:
                                print(f"   📈 RSI Positive Count: {rsi_status}/6")
                            else:
                                print(f"   ⚠️  No Relative Strength data found")
                        else:
                            print(f"   ⚠️  Failed to fetch indicators")
                    
                    # Add to financial results
                    financial_results[-1]["RSI Status"] = rsi_status
                    
                    # Add individual indicators
                    for k, v in se_data.items():
                        financial_results[-1][k] = f"{v['value']} ({v['status']})"

                else:
                    print(f"   ❌ BAD: Revenue {change_pct:+.1f}% - SKIPPED")
                    skipped_bad_status.append(symbol)
            else:
                print(f"   ⚠️  No revenue data")
                failed_stocks.append(symbol)
                
        except Exception as e:
            print(f"   ❌ Error: {str(e)[:80]}")
            failed_stocks.append(symbol)

finally:
    driver.quit()

# --------------------------
# 4️⃣ Generate Report
# --------------------------
print(f"\n{'='*70}")
print("📊 FINAL SUMMARY")
print(f"{'='*70}")
print(f"✅ Good Status (Included): {len(financial_results)}")
print(f"❌ Bad Status (Skipped):   {len(skipped_bad_status)}")
print(f"⚠️  Failed/No Data:        {len(failed_stocks)}")
print(f"📈 Total Processed:        {len(stocks_with_ids)}")

if financial_results:
    # Merge data
    financial_df = pd.DataFrame(financial_results)
    
    if delivery_results:
        delivery_df = pd.concat(delivery_results, ignore_index=True)
        
        # Clean columns
        delivery_df.columns = (delivery_df.columns.str.strip().str.upper()
                              .str.replace(" ", "_")
                              .str.replace("%", "PCT")
                              .str.replace("(", "").str.replace(")", ""))
        
        # Parse dates
        if "DATE" in delivery_df.columns:
            delivery_df["DATE"] = pd.to_datetime(delivery_df["DATE"], format="%d %b '%y", errors="coerce")
            delivery_df = delivery_df.sort_values(["STOCK", "DATE"], ascending=[True, False])
            
            # Get latest delivery
            latest_delivery = delivery_df.groupby("STOCK").nth(0).reset_index()
            
            # Merge
            financial_df.columns = financial_df.columns.str.strip().str.upper()
            
            # Find delivery columns
            delivery_cols = ["STOCK", "DATE"]
            for col in ["DELIVERY_PCT", "PRICE_CHANGE_PCT", "INSIGHT_VS_WEEKLY_AVG", "CLOSE_PRICE"]:
                matches = [c for c in latest_delivery.columns if col in c]
                if matches:
                    delivery_cols.append(matches[0])
            
            final_df = pd.merge(financial_df, latest_delivery[delivery_cols], on="STOCK", how="left")
        else:
            final_df = financial_df
    else:
        final_df = financial_df
    
    # Save to reports folder
    # timestamp = datetime.now().strftime("%Y-%m-%d")
    timestamp = datetime.now().strftime("%d-%m-%Y")
    # output_file = f"swing_screener_reports/swing_good_stocks_{timestamp}.xlsx"
    output_file = f"swing_screener_reports/{timestamp}.xlsx"

    final_df.to_excel(output_file, index=False)
    
    print(f"\n✅ Report saved: {output_file}")
    print(f"📊 Total stocks in report: {len(final_df)}")
    
    # Show preview
    if len(final_df) > 0:
        print(f"\n{'='*70}")
        print("📋 GOOD STOCKS PREVIEW:")
        print(f"{'='*70}")
        print(f"{'='*70}")
        preview_cols = ['STOCK', 'LATEST REVENUE', 'CHANGE %', 'STATUS', 'RSI STATUS']
        available_cols = [c for c in preview_cols if c in final_df.columns]
        print(final_df[available_cols].head(10).to_string(index=False))
else:
    print("\n⚠️  No stocks with Good status found.")

print(f"\n{'='*70}")
print("✅ AUTOMATION COMPLETE!")
print(f"{'='*70}")

     SWING TRADING AUTO SCREENER - FULLY DYNAMIC
📂 Loaded 251 Trendlyne IDs from cache
📂 Loaded 232 StockEdge IDs from cache
📂 Loaded 251 stock IDs from cache

📡 Fetching momentum stocks from API...
✅ Found 66 momentum stocks from API
📋 Have IDs for: 64 stocks
🔍 Need to find IDs for: 2 stocks

🌐 Setting up Chrome browser...

🔍 Finding IDs for 2 new stocks...

[1/2] HINDPETRO
   🔍 Searching for HINDPETRO...
   ✅ Found: 556/HINDPETRO/hindustan-petroleum-corporation-ltd

[2/2] TVSMOTOR
   🔍 Searching for TVSMOTOR...
   ✅ Found: 1430/TVSMOTOR/tvs-motor-company-ltd
💾 Saved 253 Trendlyne IDs to cache

✅ Now have IDs for 66 stocks

🚀 Starting to process 66 stocks...

──────────────────────────────────────────────────────────────────────
🔎 [1/66] CHOLAFIN
   ⏱️  Remaining: 65 stocks | ETA: calculating...
   📊 Checking revenue...
   ✅ GOOD: Revenue +6.9% (7985 → 8539)
   📦 Getting delivery data...
   ✅ Delivery data collected
   📊 Getting StockEdge indicators...
   📈 RSI Positive Count: 6/6

──